In [3]:
import pandas as pd
import numpy as np
data = pd.read_csv("abalone.data", header=None)


data.columns=["Sex", "Length", "Diameter", "Height", "Whole weight",
    "Shucked weight", "Viscera weight", "Shell weight", "Rings"
]

In [4]:
data = pd.get_dummies(data, columns=["Sex"], drop_first=True)

In [5]:
data["Age"] = data["Rings"] + 1.5
data.drop("Rings", axis=1, inplace=True)

# Split features and target
X = data.drop("Age", axis=1).values
y = data["Age"].values.reshape(-1, 1)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
class MLP:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return z > 0

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        return self.z2

    def backward(self, X, y, y_pred):
        m = X.shape[0]

        dZ2 = y_pred - y
        dW2 = (1/m) * self.a1.T @ dZ2
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_derivative(self.z1)
        dW1 = (1/m) * X.T @ dZ1
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            self.backward(X, y, y_pred)

            if epoch % 100 == 0:
                loss = np.mean((y_pred - y) ** 2)
                print(f"Epoch {epoch}, MSE: {loss:.4f}")


In [9]:
mlp = MLP(input_size=X_train.shape[1], hidden_size=16, output_size=1, lr=0.01)
mlp.train(X_train, y_train, epochs=1000)


Epoch 0, MSE: 141.2686
Epoch 100, MSE: 6.5558
Epoch 200, MSE: 5.1273
Epoch 300, MSE: 4.9287
Epoch 400, MSE: 4.8826
Epoch 500, MSE: 4.8637
Epoch 600, MSE: 4.8493
Epoch 700, MSE: 4.8364
Epoch 800, MSE: 4.8242
Epoch 900, MSE: 4.8125


In [10]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = mlp.forward(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error:", mae)
print("R² Score:", r2)


Mean Absolute Error: 1.6000983221924092
R² Score: 0.5492402747145347


In [11]:
y_pred_rounded = np.round(y_pred)
y_test_rounded = np.round(y_test)

accuracy = np.mean(y_pred_rounded == y_test_rounded)
print("Rounded Age Accuracy:", accuracy)


Rounded Age Accuracy: 0.21889952153110048
